In [3]:
import pandas as pd
import numpy as np
import os

# ================= 1. 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_PATH = "primekg_ppmi_base_unrefined.csv"

# 1. 黑名单 (保持不变，修正了原代码中多余的字母'A')
BLACKLIST_NODES = [
    "multi-cellular organism", "blood", "small intestine", "intestine", "colon", "esophagus", "stomach", 
    "testis", "fallopian tube", "prostate gland", "myometrium", "female gonad", "endometrium", 
    "spleen", "lymph node", "adrenal gland", "thyroid gland", "pancreas", 
    "kidney", "adult mammalian kidney", "cortex of kidney", "urinary bladder", 
    "heart", "heart left ventricle", "cardiac muscle", "lung", "muscle of leg", 
    "saliva-secreting gland",
    "cytoplasm", "cytosol", "nucleus", "membrane", "plasma membrane", 
    "extracellular region", "extracellular space", "mitochondrion",
    "protein binding", "molecular_function", "biological_process", "cellular_component",
    "cell surface", "intracellular", "nucleoplasm"
]

# 2. 定义两组关键词，用于提取相关领域知识
KEYWORDS_PATHOLOGY = [
    "Parkinson", "Dopamine", "Synuclein", "Lewy", "Neurodegeneration",
    "Tremor", "Rigidity", "Bradykinesia", "Hypokinesia", 
    "Gait", "Posture", "Instability", "Freezing", "Festinating", 
    "Speech", "Dysarthria", "Facial", "Hypomimia"
]

KEYWORDS_ANATOMY = [
    "Substantia nigra", "Striatum", "Putamen", "Caudate", "Globus pallidus", "Basal ganglia"
]

def main():
    print(f"🔹 1. 加载 PrimeKG...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except:
        print("❌ 找不到文件: kg.csv")
        return

    # --- 清洗 ---
    print(f"🔹 2. 执行基础清洗...")
    valid_types = ['disease', 'gene/protein', 'effect/phenotype', 'pathway', 'biological_process', 'anatomy']
    mask = (
        (df['x_type'].isin(valid_types)) & (df['y_type'].isin(valid_types)) &
        (~df['relation'].isin(['indication', 'contraindication', 'drug_drug', 'off-label use'])) &
        (~df['x_name'].isin(BLACKLIST_NODES)) & (~df['y_name'].isin(BLACKLIST_NODES))
    )
    df_clean = df[mask].copy()

    # --- 核心逻辑退化: 直接提取相关领域知识，不做交集过滤 ---
    print(f"🔹 3. 执行【领域提取】(未精炼：保留解剖节点的全部原始连接)...")
    
    # 1. 提取病理子图 (Group A)
    pat_A = '|'.join(KEYWORDS_PATHOLOGY)
    mask_A = (df_clean['x_name'].astype(str).str.contains(pat_A, case=False, na=False)) | \
             (df_clean['y_name'].astype(str).str.contains(pat_A, case=False, na=False))
    df_pathology = df_clean[mask_A].copy()
    print(f"   - 病理/症状相关边: {len(df_pathology)} 条")
    
    # 2. 提取解剖锚点 (Group B) - 【修改点：直接提取，不强制要求另一端与PD相关】
    pat_B = '|'.join(KEYWORDS_ANATOMY)
    mask_B_unrefined = (
        (df_clean['x_name'].astype(str).str.contains(pat_B, case=False, na=False)) |
        (df_clean['y_name'].astype(str).str.contains(pat_B, case=False, na=False))
    )
    df_anatomy_unrefined = df_clean[mask_B_unrefined].copy()
    print(f"   - 解剖相关边 (未过滤通用连接): {len(df_anatomy_unrefined)} 条")
    
    # 3. 合并核心层
    core_df = pd.concat([df_pathology, df_anatomy_unrefined]).drop_duplicates()
    print(f"   ✅ 提取的领域知识: {len(core_df)} 条")

    # --- PPI 补充 ---
    print(f"\n🔹 4. 补充骨架 PPI (【修改点：全量引入，不剪枝】)...")
    genes = set(core_df[core_df['y_type'] == 'gene/protein']['y_name']) | \
            set(core_df[core_df['x_type'] == 'gene/protein']['x_name'])
            
    mask_ppi = (
        (df_clean['x_name'].isin(genes)) & (df_clean['y_name'].isin(genes)) &
        (df_clean['relation'] == 'protein_protein')
    )
    ppi_full = df_clean[mask_ppi].copy()
    print(f"   - 全量 PPI 关系: {len(ppi_full)} 条")

    # --- 保存 ---
    final_df = pd.concat([core_df, ppi_full]).drop_duplicates()
    print(f"\n🔹 5. 保存未精炼版底座至: {OUTPUT_PATH}")
    final_df.to_csv(OUTPUT_PATH, index=False)
    print("-" * 50)
    print(f"🎉 最终条目数: {len(final_df)} (预期显著大于精炼版的 3.2w)")
    print("-" * 50)

if __name__ == "__main__":
    main()

🔹 1. 加载 PrimeKG...
🔹 2. 执行基础清洗...
🔹 3. 执行【领域提取】(未精炼：保留解剖节点的全部原始连接)...
   - 病理/症状相关边: 26212 条
   - 解剖相关边 (未过滤通用连接): 105456 条
   ✅ 提取的领域知识: 131554 条

🔹 4. 补充骨架 PPI (【修改点：全量引入，不剪枝】)...
   - 全量 PPI 关系: 407088 条

🔹 5. 保存未精炼版底座至: primekg_ppmi_base_unrefined.csv
--------------------------------------------------
🎉 最终条目数: 538642 (预期显著大于精炼版的 3.2w)
--------------------------------------------------


In [4]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from collections import Counter

# ================= 1. 配置区 =================
# PPMI 本地数据文件 (对应你的 control, PD1, prodromal, swedd)
PPMI_FILES = ['control.csv', 'PD1.csv', 'prodromal.csv', 'swedd.csv']

# 指向 Step 1 生成的未精炼底座
PRIMEKG_PATH = "primekg_ppmi_base_unrefined.csv"

# 输出文件 (添加 _unrefined 后缀)
OUTPUT_TRIPLETS = "ppmi_knowledge_triplets_unrefined.csv"
OUTPUT_E2ID = "ppmi_kg_entity2id_unrefined.json"
OUTPUT_R2ID = "ppmi_kg_relation2id_unrefined.json"

# ================= 2. 映射逻辑定义 =================

# 基础人口学映射
base_col_map = {
    "Age": "Concept:Age",
    "Sex": "Concept:Sex",
    "Group": "Concept:Group"  # PD, Control, Prodromal 等
}

# [PPMI特有] 症状映射 -> 对应 PrimeKG 里的关键词
# 逻辑：UPDRS 列分值 > 0，则连接到 PrimeKG 里的对应节点
# 注意：右边的关键词必须是 Step 2 中确认保留的词
# ================= 修正后的 SYMPTOM_MAP =================
SYMPTOM_MAP = {
    "Tremor": ([
        "code_upd2315a_postural_tremor_of_right_hand", "code_upd2315b_postural_tremor_of_left_hand",
        "code_upd2316a_kinetic_tremor_of_right_hand", "code_upd2316b_kinetic_tremor_of_left_hand",
        "code_upd2317a_rest_tremor_amplitude_right_upper_extremity", "code_upd2317b_rest_tremor_amplitude_left_upper_extremity",
        "code_upd2317c_rest_tremor_amplitude_right_lower_extremity", "code_upd2317d_rest_tremor_amplitude_left_lower_extremity",
        "code_upd2317e_rest_tremor_amplitude_lip_or_jaw"
    ], "Tremor"), 

    "Rigidity": ([
        "code_upd2303a_rigidity_neck", "code_upd2303b_rigidity_rt_upper_extremity",
        "code_upd2303c_rigidity_left_upper_extremity", "code_upd2303d_rigidity_rt_lower_extremity",
        "code_upd2303e_rigidity_left_lower_extremity"
    ], "Rigidity"),

    "Bradykinesia": ([
        "code_upd2304a_right_finger_tapping", "code_upd2304b_left_finger_tapping",
        "code_upd2305a_right_hand_movements", "code_upd2305b_left_hand_movements",
        "code_upd2306a_pron_sup_movement_right_hand", "code_upd2306b_pron_sup_movement_left_hand",
        "code_upd2307a_right_toe_tapping", "code_upd2307b_left_toe_tapping",
        "code_upd2308a_right_leg_agility", "code_upd2308b_left_leg_agility",
        "code_upd2314_body_bradykinesia"
    ], "Bradykinesia"),

    # [修正 1] Gait -> 改为更通用的 Gait disturbance，防止全部被映射为 Freezing
    "Gait": ([
        "code_upd2309_arising_from_chair", "code_upd2310_gait",
        "code_upd2311_freezing_of_gait", "code_upd2312_postural_stability", "code_upd2313_posture"
    ], "Gait"), # 搜索 Gait
    
    # [修正 2] Facial -> 既然 Hypomimia 没找到，尝试 Facies (面容)，如果还没找到，代码里会fallback到 Bradykinesia
    "Facial": (["code_upd2302_facial_expression"], "Facies"), 
    
    # [修正 3] Speech -> 改为 Dysarthria (构音障碍)，原先 Absent speech (哑) 太重了
    "Speech": (["code_upd2301_speech_problems"], "Dysarthria")
}

# ================= 修正后的 find_best_kg_match 函数 =================
def find_best_kg_match(keyword):
    """在底座中寻找最匹配的节点名 (增强版)"""
    kw_lower = keyword.lower()
    
    # 1. 精确匹配
    if kw_lower in VALID_KG_NODES:
        return f"PrimeKG:{keyword}" 
    
    # 2. 模糊匹配 (找包含关键词的第一个节点)
    # 优先找短的词，防止匹配到 "Gait analysis" 这种不相关的
    matches = []
    for node in VALID_KG_NODES:
        if kw_lower in node:
            matches.append(node)
    
    if matches:
        # 按长度排序，找最短的匹配项 (通常是最核心的概念)
        # 比如搜 "Gait"，优先选 "Gait"，而不是 "Gait apraxia"
        best_match = min(matches, key=len)
        return f"PrimeKG:{best_match.capitalize()}"

    # 3. [新增] 兜底策略 (Fallback)
    # 如果特定词找不到，映射到其上位概念
    if keyword == "Facies": # 如果 Facies (面容) 也没找到
        print("         [Fallback] 'Facies' 未找到，回退映射至 'Bradykinesia' (Facial Bradykinesia)")
        return find_best_kg_match("Bradykinesia") # 面具脸本质是面部少动
            
    return None
# 全局缓存
VALID_KG_NODES = set()

def load_primekg():
    """读取筛选后的 PrimeKG 生物底座"""
    print(f"🔹 1. 正在读取 PrimeKG 生物底座: {PRIMEKG_PATH} ...")
    if not os.path.exists(PRIMEKG_PATH):
        print(f"❌ 错误：找不到 {PRIMEKG_PATH}，请先运行 Step 2 代码")
        return []

    triplets = []
    global VALID_KG_NODES
    
    try:
        df = pd.read_csv(PRIMEKG_PATH, low_memory=False)
        
        cols = df.columns
        x_col = next((c for c in cols if 'x_name' in c or 'head' in c), 'x_name')
        y_col = next((c for c in cols if 'y_name' in c or 'tail' in c), 'y_name')
        r_col = next((c for c in cols if 'relation' in c), 'relation')

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing PrimeKG"):
            h_raw = str(row[x_col])
            t_raw = str(row[y_col])
            
            h = "PrimeKG:" + h_raw
            t = "PrimeKG:" + t_raw
            r = str(row[r_col])
            
            triplets.append((h, r, t))
            
            VALID_KG_NODES.add(h_raw.lower())
            VALID_KG_NODES.add(t_raw.lower())

        print(f"   ✅ 已加载生物医学知识: {len(triplets)} 条")
        
    except Exception as e:
        print(f"   ❌ 读取 PrimeKG 失败: {e}")
    return triplets



def process_ppmi_files():
    """处理 PPMI CSV 文件"""
    print(f"🔹 2. 正在处理 PPMI 临床数据...")
    local_triplets = []
    patient_ids = set()
    
    stats = {
        "patients": 0,
        "symptoms_links": 0  # 记录建立了多少条症状连接
    }
    
    # 预先找到 PrimeKG 里的对应节点，避免每行都搜一遍
    group_target_nodes = {}
    print("      -> 正在预检 PrimeKG 节点匹配情况...")
    for label, (_, keyword) in SYMPTOM_MAP.items():
        match = find_best_kg_match(keyword)
        if match:
            group_target_nodes[label] = match
            print(f"         [OK] '{label}' 映射到 -> {match}")
        else:
            print(f"         [⚠️] 警告: 底座里没找到 '{keyword}'，该症状将无法连接！")

    for file_name in PPMI_FILES:
        if not os.path.exists(file_name):
            continue

        print(f"      -> 正在解析: {file_name} ...")
        df = pd.read_csv(file_name)

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"   Reading {file_name}"):
            # 1. 构建病人 ID
            if 'Subject' in row and pd.notna(row['Subject']):
                p_id_raw = str(int(row['Subject']))
                patient_id = f"Patient:{p_id_raw}"
                patient_ids.add(patient_id)
                stats["patients"] += 1
            else:
                continue

            # 2. 基础属性 (Age, Sex, Group)
            for col, relation_name in base_col_map.items():
                if col in row and pd.notna(row[col]):
                    val = str(row[col])
                    target_node = f"{relation_name}:{val}"
                    local_triplets.append((patient_id, "has_attribute", target_node))

            # 3. UPDRS 症状映射 -> 动态连接 PrimeKG
            for label, (cols, _) in SYMPTOM_MAP.items():
                target_node = group_target_nodes.get(label)
                if not target_node: continue
                
                # 只要该组有一个列的分数 > 0，就视为有该症状
                has_symptom = False
                for c in cols:
                    if c in row and pd.notna(row[c]) and row[c] > 0:
                        has_symptom = True
                        break
                
                if has_symptom:
                    # 建立连接：Patient -> has_symptom -> PrimeKG:Tremor
                    local_triplets.append((patient_id, "has_symptom", target_node))
                    stats["symptoms_links"] += 1

    print(f"   ✅ PPMI 处理完成: 生成 {len(local_triplets)} 条临床关系")
    print(f"      统计摘要:")
    print(f"      - 解析病人: {stats['patients']}")
    print(f"      - 成功构建症状连接: {stats['symptoms_links']} 条 (Bridge to PrimeKG)")
    
    return local_triplets

def analyze_graph_statistics(df):
    """打印详细的图谱体检报告 (PPMI版)"""
    print("\n" + "="*60)
    print("📊 [最终 PPMI 异构图谱深度体检报告]")
    print("="*60)
    
    # 1. 节点类型分析
    all_nodes = list(df['head']) + list(df['tail'])
    unique_nodes = set(all_nodes)
    
    node_types = []
    for n in unique_nodes:
        n_str = str(n)
        if n_str.startswith("Patient:"): node_types.append("Patient (病人)")
        elif n_str.startswith("PrimeKG:"): node_types.append("PrimeKG (生物知识)")
        elif n_str.startswith("Concept:"): node_types.append("Concept (人口学)")
        else: node_types.append("Other (其他)")
        
    type_counts = Counter(node_types)
    print(f"1. 节点构成 (总数: {len(unique_nodes)}):")
    for ntype, count in type_counts.most_common():
        print(f"   - {ntype:<20} : {count}")
        
    # 2. 关系分布
    print(f"\n2. 关系分布 (Top 15):")
    print(df['relation'].value_counts().head(15))
    
    # 3. 桥梁检查 (PPMI 特别版)
    print(f"\n3. 关键连接检查:")
    
    # 检查 病人 -> PrimeKG 的连接
    symptom_mask = (df['relation'] == 'has_symptom') & (df['tail'].str.startswith("PrimeKG:"))
    symptom_count = len(df[symptom_mask])
    print(f"   - 病人 -> PrimeKG症状节点 (Tremor等) 连接数: {symptom_count} 条 (核心指标!)")

    print("="*60 + "\n")

def main():
    # 1. 加载底座
    kg_triplets = load_primekg()
    
    # 2. 处理临床数据 (会利用底座信息进行匹配)
    ppmi_triplets = process_ppmi_files()

    if not ppmi_triplets:
        print("❌ 严重警告：未生成任何 PPMI 临床数据！")
        return

    # 3. 合并
    all_triplets = kg_triplets + ppmi_triplets

    # 4. 保存与去重
    df_out = pd.DataFrame(all_triplets, columns=['head', 'relation', 'tail'])

    before_dedup = len(df_out)
    df_out.drop_duplicates(inplace=True)
    after_dedup = len(df_out)

    print(f"✂️  执行去重操作: {before_dedup} -> {after_dedup} (移除 {before_dedup - after_dedup})")

    # 5. 体检
    analyze_graph_statistics(df_out)

    # 6. 保存
    df_out.to_csv(OUTPUT_TRIPLETS, index=False)
    print(f"💾 最终 PPMI 异构图谱已保存至: {OUTPUT_TRIPLETS}")

    # 7. 生成 ID 映射
    print("🔹 正在生成 Entity/Relation ID 映射表...")
    entities = sorted(list(set(df_out['head']) | set(df_out['tail'])))
    relations = sorted(list(set(df_out['relation'])))

    ent2id = {e: i for i, e in enumerate(entities)}
    rel2id = {r: i for i, r in enumerate(relations)}

    with open(OUTPUT_E2ID, 'w') as f: json.dump(ent2id, f)
    with open(OUTPUT_R2ID, 'w') as f: json.dump(rel2id, f)

    print(f"   映射表已保存: {OUTPUT_E2ID}, {OUTPUT_R2ID}")
    
    # 验证一个 ID (来自你的 PD1.csv 样例)
    sample_id = "Patient:3102" 
    count = sum(1 for e in entities if sample_id in e)
    print(f"\n🔍 验证环节: 检查是否有示例病人 ({sample_id})")
    print(f"   包含该病人? {'✅ 是' if count > 0 else '❌ 否'}")

if __name__ == "__main__":
    main()

🔹 1. 正在读取 PrimeKG 生物底座: primekg_ppmi_base_unrefined.csv ...


Parsing PrimeKG: 100%|██████████████████████████████████████████████████████| 538642/538642 [00:45<00:00, 11910.73it/s]


   ✅ 已加载生物医学知识: 538642 条
🔹 2. 正在处理 PPMI 临床数据...
      -> 正在预检 PrimeKG 节点匹配情况...
         [OK] 'Tremor' 映射到 -> PrimeKG:Tremor
         [OK] 'Rigidity' 映射到 -> PrimeKG:Rigidity
         [OK] 'Bradykinesia' 映射到 -> PrimeKG:Bradykinesia
         [OK] 'Gait' 映射到 -> PrimeKG:Gait ataxia
         [OK] 'Facial' 映射到 -> PrimeKG:Moon facies
         [OK] 'Speech' 映射到 -> PrimeKG:Dysarthria
      -> 正在解析: control.csv ...


   Reading control.csv: 100%|██████████████████████████████████████████████████████| 132/132 [00:00<00:00, 2005.46it/s]


      -> 正在解析: PD1.csv ...


...

      -> 正在解析: prodromal.csv ...


   Reading prodromal.csv: 100%|██████████████████████████████████████████████████████| 80/80 [00:00<00:00, 2186.44it/s]


      -> 正在解析: swedd.csv ...


   Reading swedd.csv: 100%|██████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 1883.31it/s]


   ✅ PPMI 处理完成: 生成 5353 条临床关系
      统计摘要:
      - 解析病人: 804
      - 成功构建症状连接: 2941 条 (Bridge to PrimeKG)
✂️  执行去重操作: 543995 -> 541382 (移除 2613)

📊 [最终 PPMI 异构图谱深度体检报告]
1. 节点构成 (总数: 18171):
   - PrimeKG (生物知识)       : 17775
   - Patient (病人)         : 339
   - Concept (人口学)        : 57

2. 关系分布 (Top 15):
relation
protein_protein               407088
anatomy_protein_present       103962
disease_phenotype_positive     20606
disease_protein                 2204
disease_disease                 1718
has_attribute                   1474
has_symptom                     1270
phenotype_phenotype              772
bioprocess_protein               706
phenotype_protein                432
bioprocess_bioprocess            354
anatomy_protein_absent           344
anatomy_anatomy                  244
disease_phenotype_negative       122
pathway_protein                   72
Name: count, dtype: int64

3. 关键连接检查:
   - 病人 -> PrimeKG症状节点 (Tremor等) 连接数: 1270 条 (核心指标!)

💾 最终 PPMI 异构图谱已保存至: ppmi_knowledge_trip

In [7]:
# 知识图谱剪枝与优化 (PPMI未精炼版)
import pandas as pd
import networkx as nx
import os
import shutil

# ================= ⚡ 配置区 =================
# 目标文件 (对应 Step 2 的输出，已添加 _unrefined 后缀)
TARGET_FILE = 'ppmi_knowledge_triplets_unrefined.csv'

# 保护名单：这些前缀的节点绝对不能删，哪怕它们只有一条连线
# 确保所有病人节点和临床属性不被误杀
PROTECTED_PREFIXES = [
    "Patient:",   # 病人 (核心)
    "Concept:",   # 年龄/性别/分组
    "PrimeKG:Tremor", # 显式保护核心症状节点
    "PrimeKG:Rigidity",
    "PrimeKG:Bradykinesia",
    "PrimeKG:Gait",
    "PrimeKG:Dysarthria",
    "PrimeKG:Putamen", # 显式保护解剖锚点
    "PrimeKG:Substantia nigra"
]

def analyze_graph(df, stage="清洗前"):
    """📊 打印图谱健康状况"""
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # 计算连通子图数量 (理想情况下应该很少，说明知识都连在一起)
    components = list(nx.connected_components(G))
    num_components = len(components)
    largest_comp = len(max(components, key=len)) if components else 0
    
    print(f"--- 🏥 [{stage}] 体检报告 ---")
    print(f"   节点: {num_nodes} | 边: {num_edges}")
    print(f"   连通孤岛数: {num_components} (越少越好)")
    print(f"   最大连通子图节点数: {largest_comp}")
    return G

def main():
    if not os.path.exists(TARGET_FILE):
        print(f"❌ 找不到文件: {TARGET_FILE}")
        return

    print(f"🚀 正在加载 {TARGET_FILE} ...")
    try:
        df = pd.read_csv(TARGET_FILE)
    except Exception as e:
        print(f"❌ 读取失败: {e}")
        return
    
    # 1. 术前体检
    analyze_graph(df, "清洗前")
    
    # 2. 创建备份 (安全第一)
    backup_file = TARGET_FILE + ".bak"
    shutil.copy(TARGET_FILE, backup_file)
    print(f"📦 已创建备份文件: {backup_file} (如果出问题可以用这个恢复)")

    # ================= 核心清洗逻辑 =================
    print("\n🧹 开始执行原地净化...")
    
    # 构建图结构
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    nodes_to_keep = set()
    
    # --- 策略 A: 移除无法到达病人的“死知识” ---
    # 逻辑：如果一个连通子图里连一个病人都没有，那这个子图就是垃圾数据
    print("   1. 正在切除与病人断连的孤立知识岛屿...")
    components = nx.connected_components(G)
    
    removed_islands = 0
    for comp in components:
        # 检查这个岛屿里有没有病人
        has_patient = any(str(n).startswith("Patient:") for n in comp)
        # 检查有没有关键解剖结构，防止误删 (虽然理论上解剖结构都该连着)
        has_anatomy = any("Substantia nigra" in str(n) for n in comp)
        
        if has_patient or has_anatomy:
            nodes_to_keep.update(comp)
        else:
            removed_islands += 1
            
    print(f"      -> 移除了 {removed_islands} 个无效孤岛")
            
    # --- 策略 B: 修剪末梢悬挂节点 (Dead Ends) ---
    print("   2. 正在修剪低效的末梢节点...")
    # 只在保留下来的节点里修剪
    G_sub = G.subgraph(list(nodes_to_keep)).copy()
    
    nodes_to_remove = []
    for node in G_sub.nodes():
        node_str = str(node)
        # 如果是受保护节点，跳过
        if any(node_str.startswith(p) for p in PROTECTED_PREFIXES):
            continue
        # 针对核心症状的模糊保护
        if "Tremor" in node_str or "Rigidity" in node_str:
            continue
            
        # 如果度为1 (悬挂)，且不是受保护节点 -> 视为噪声
        # 这通常是 PrimeKG 里某个偏僻的蛋白，只连了一个上级，没连其他任何东西
        if G_sub.degree(node) == 1:
            nodes_to_remove.append(node)
            
    final_nodes = set(G_sub.nodes()) - set(nodes_to_remove)
    print(f"      -> 标记了 {len(nodes_to_remove)} 个悬挂节点准备删除")
    
    # ================= 保存结果 =================
    print("   3. 正在重组数据...")
    
    # 筛选只包含 final_nodes 的三元组
    mask = df['head'].isin(final_nodes) & df['tail'].isin(final_nodes)
    df_clean = df[mask].copy()
    
    # 3. 术后体检
    analyze_graph(df_clean, "清洗后")
    
    # 覆盖原文件
    df_clean.to_csv(TARGET_FILE, index=False)
    print(f"\n✅ 成功！原文件 {TARGET_FILE} 已更新。")
    print(f"   (原数据量 {len(df)} -> 现数据量 {len(df_clean)})")
    print("👉 你现在可以运行 DistMult 训练脚本了！")

if __name__ == "__main__":
    main()

🚀 正在加载 ppmi_knowledge_triplets_unrefined.csv ...
--- 🏥 [清洗前] 体检报告 ---
   节点: 15558 | 边: 269509
   连通孤岛数: 1 (越少越好)
   最大连通子图节点数: 15558
📦 已创建备份文件: ppmi_knowledge_triplets_unrefined.csv.bak (如果出问题可以用这个恢复)

🧹 开始执行原地净化...
   1. 正在切除与病人断连的孤立知识岛屿...
      -> 移除了 0 个无效孤岛
   2. 正在修剪低效的末梢节点...
      -> 标记了 33 个悬挂节点准备删除
   3. 正在重组数据...
--- 🏥 [清洗后] 体检报告 ---
   节点: 15525 | 边: 269476
   连通孤岛数: 1 (越少越好)
   最大连通子图节点数: 15525

✅ 成功！原文件 ppmi_knowledge_triplets_unrefined.csv 已更新。
   (原数据量 536286 -> 现数据量 536220)
👉 你现在可以运行 DistMult 训练脚本了！


In [9]:
# train_ppmi_distmult.py
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
import json
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

# ================= ⚡ 配置区 (PPMI 版) =================
# 1. 输入文件
TRIPLETS_FILE = 'ppmi_knowledge_triplets_unrefined.csv'

# 2. 映射表输出
ENTITY2ID_FILE = 'ppmi_kg_entity2id_unrefined.json'
RELATION2ID_FILE = 'ppmi_kg_relation2id_unrefined.json'

# 3. 输出文件
OUTPUT_EMBED = 'ppmi_kg_embeddings_unrefined.npy'

# 4. 训练超参数 (PPMI 适配)
EMBED_DIM = 128
NUM_EPOCHS = 20   # 小图谱建议多训练几轮
BATCH_SIZE = 512
LR = 0.001         # 降低学习率求稳
TRAIN_RATIO = 0.95 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 [PPMI] 训练设备: {DEVICE}")

# ================= 🛠️ 辅助函数: 重建 ID 映射 =================
def rebuild_mappings(df):
    print("🔄 正在基于最终数据重建 ID 映射表...")
    entities = sorted(list(set(df['head']) | set(df['tail'])))
    relations = sorted(list(set(df['relation'])))
    
    ent2id = {e: i for i, e in enumerate(entities)}
    rel2id = {r: i for i, r in enumerate(relations)}
    
    print(f"   - 实体总数: {len(entities)}")
    print(f"   - 关系总数: {len(relations)}")
    
    with open(ENTITY2ID_FILE, 'w') as f: json.dump(ent2id, f)
    with open(RELATION2ID_FILE, 'w') as f: json.dump(rel2id, f)
    
    return ent2id, rel2id

# ================= 🛠️ 数据加载类 =================
class KGDataset(Dataset):
    def __init__(self, df, entity2id, relation2id):
        self.triplets = []
        skipped = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Indexing Data"):
            try:
                h = entity2id.get(str(row['head']).strip())
                r = relation2id.get(str(row['relation']).strip())
                t = entity2id.get(str(row['tail']).strip())
                if h is not None and r is not None and t is not None:
                    self.triplets.append((h, r, t))
                else:
                    skipped += 1
            except Exception:
                skipped += 1
                continue
        self.triplets = torch.LongTensor(self.triplets)
        print(f"    📊 最终有效三元组: {len(self.triplets)}")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        return self.triplets[idx]

# ================= 🧠 DistMult 模型 =================
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embed_dim):
        super(DistMult, self).__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        self.criterion = nn.MarginRankingLoss(margin=1.0)

    def forward(self, h, r, t):
        h_e = self.ent_emb(h)
        r_e = self.rel_emb(r)
        t_e = self.ent_emb(t)
        score = torch.sum(h_e * r_e * t_e, dim=1)
        return score

    def calculate_loss(self, h, r, t):
        batch_size = h.size(0)
        neg_t = torch.randint(0, self.num_entities, (batch_size,), device=h.device)
        pos_score = self.forward(h, r, t)
        neg_score = self.forward(h, r, neg_t)
        target = torch.ones(batch_size, device=h.device)
        loss = self.criterion(pos_score, neg_score, target)
        return loss

# ================= 🏃 主训练循环 =================
def train():
    if not os.path.exists(TRIPLETS_FILE):
        print(f"❌ 找不到 {TRIPLETS_FILE}，请检查路径")
        return

    # 1. 读取与映射
    print(f"📥 正在读取: {TRIPLETS_FILE} ...")
    df = pd.read_csv(TRIPLETS_FILE)
    entity2id, relation2id = rebuild_mappings(df)
    
    num_ents = len(entity2id)
    num_rels = len(relation2id)

    # 2. Dataset与Loader
    dataset = KGDataset(df, entity2id, relation2id)
    train_size = int(TRAIN_RATIO * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # 3. 初始化
    model = DistMult(num_ents, num_rels, EMBED_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\n🔥 [PPMI] 开始训练 (共 {NUM_EPOCHS} 轮)...")

    # 4. 训练循环 (逻辑已还原：每轮都打印)
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=False)

        for batch in progress:
            batch = batch.to(DEVICE)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

            optimizer.zero_grad()
            loss = model.calculate_loss(h, r, t)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # [修正] 恢复每轮都验证的逻辑
        if (epoch + 1) % 1 == 0:
            model.eval()
            test_loss = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(DEVICE)
                    h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
                    loss = model.calculate_loss(h, r, t)
                    test_loss += loss.item()
            
            avg_test = test_loss / len(test_loader) if len(test_loader) > 0 else 0
            # 打印格式保持一致
            print(f"Epoch {epoch + 1:03d} | 📉 Train Loss: {avg_loss:.4f} | 🔍 Test Loss: {avg_test:.4f}")

    # 5. 保存
    print("\n" + "=" * 40)
    print(f"💾 训练完成！正在保存 PPMI Embeddings 至: {OUTPUT_EMBED}")

    embeddings = model.ent_emb.weight.detach().cpu().numpy()
    np.save(OUTPUT_EMBED, embeddings)

    print(f"✅ 保存成功！矩阵形状: {embeddings.shape}")
    print(f"   (行数应为 {num_ents}，列数应为 {EMBED_DIM})")
    print("=" * 40)
    print("👉 下一步：多模态分类任务。")
    print("   记得加载 ppmi_kg_entity2id_unrefined.json。")

if __name__ == "__main__":
    train()

🚀 [PPMI] 训练设备: cpu
📥 正在读取: ppmi_knowledge_triplets_unrefined.csv ...
🔄 正在基于最终数据重建 ID 映射表...
   - 实体总数: 15525
   - 关系总数: 16


Indexing Data: 100%|████████████████████████████████████████████████████████| 536220/536220 [00:38<00:00, 13917.37it/s]


    📊 最终有效三元组: 536220

🔥 [PPMI] 开始训练 (共 20 轮)...


Epoch 001 | 📉 Train Loss: 0.6896 | 🔍 Test Loss: 0.3813


Epoch 002 | 📉 Train Loss: 0.2953 | 🔍 Test Loss: 0.2793


...

Epoch 016 | 📉 Train Loss: 0.1037 | 🔍 Test Loss: 0.1860


Epoch 017 | 📉 Train Loss: 0.1029 | 🔍 Test Loss: 0.1815


Epoch 018 | 📉 Train Loss: 0.1019 | 🔍 Test Loss: 0.1883


Epoch 019 | 📉 Train Loss: 0.1004 | 🔍 Test Loss: 0.1870


Epoch 020 | 📉 Train Loss: 0.1009 | 🔍 Test Loss: 0.1836

💾 训练完成！正在保存 PPMI Embeddings 至: ppmi_kg_embeddings_unrefined.npy
✅ 保存成功！矩阵形状: (15525, 128)
   (行数应为 15525，列数应为 128)
👉 下一步：多模态分类任务。
   记得加载 ppmi_kg_entity2id_unrefined.json。
